# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [1]:
# !pip install tensorboardX
!pip install datasets
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm
from datasets import load_dataset

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 128

np.random.seed(42)
torch.manual_seed(42)
tf = transforms.Compose(
    [transforms.ToTensor()]
)
def collate(batch):
    imgs = torch.stack([tf(b["img"]) for b in batch])
    labels = torch.tensor([b["label"] for b in batch])
    return imgs, labels

batch_size = 64

ds = load_dataset("uoft-cs/cifar10")

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [2]:
train_loader = torch.utils.data.DataLoader(ds["train"].select(range(20_000)), batch_size=64, shuffle=True, collate_fn = collate)
test_loader = torch.utils.data.DataLoader(ds["test"], batch_size=64, shuffle=False, collate_fn = collate)

In [3]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs

class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)

no input normalization


# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

In [4]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    training = model.training
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        # TODO: Calculate the loss
        loss = ce_loss(output, labels)
        loss.backward()
        # TODO: compute the adv_x
        # find delta, clamp with eps
        with torch.no_grad():
          grad = adv_x.grad
          adv_x_step = adv_x + eps_step * torch.sign(grad) # Calc grad
          adv_x = torch.clamp(adv_x_step, x - eps, x + eps) # Clip
          adv_x = torch.clamp(adv_x, 0, 1) # Image clip
    if(training):
        model.train()
    return adv_x

In [5]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    training = model.training
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        batch_size = x.size()[0]
        # TODO: Calculate the loss
        loss = ce_loss(output, labels)
        loss.backward()
        grad = adv_x.grad.data
        # TODO: compute the adv_x
        # find delta, clamp with eps, project delta to the l2 ball
        # HINT: https://github.com/Harry24k/adversarial-attacks-pytorch/blob/master/torchattacks/attacks/pgdl2.py
        with torch.no_grad():
          # Take our step
          grad_norm = adv_x.grad/(torch.linalg.vector_norm(adv_x.grad, ord = 2, dim = (1, 2, 3), keepdim = True) + 1e-10)
          adv_x_step = adv_x + eps_step * (grad_norm) # The added perturbation
          # Check which of these need to be projected back to our eps ball
          diff = adv_x_step - x
          diff_norm = torch.linalg.vector_norm(diff, ord = 2, dim = (1, 2, 3), keepdim = True)
          mask_greater = (diff_norm > eps).reshape(-1)
          # Project those differences down
          diff[mask_greater] = diff[mask_greater]/torch.linalg.vector_norm(diff[mask_greater], ord = 2, dim = (1, 2, 3), keepdim = True) * eps
          adv_x = x + diff
          adv_x = torch.clamp(adv_x, 0, 1) # Image clip
    if(training):
        model.train()
    return adv_x

# Implement robust training and normal training

In [6]:
# Train robust model
import random
def train_model(model, attack, epochs, prob, k, eps, eps_step, trainloader):
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=0.001)
  path = f"/content/{attack}_model.pt"
  # Then we must add some "adversarial examples" to our training...
  # We acc need to mutate x in every 10 batches... to get percentage...
  for epoch in range(epochs):  # loop over the dataset multiple times
    model.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
      # get the inputs; data is a list of [inputs, labels]
      inputs, labels = data
      inputs = inputs.to(device)
      labels = labels.to(device)
      # With xyz percentage, mutate them
      if(attack == "pgd_linf"):
        if(random.random() < prob):
          inputs = pgd_linf_untargeted(model, inputs, labels, k, eps, eps_step)
      elif(attack == "pgd_l2"):
        if(random.random() < prob):
          inputs = pgd_l2_untargeted(model, inputs, labels, k, eps, eps_step)

      # zero the parameter gradients
      optimizer.zero_grad()

      # forward + backward + optimize
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      # print statistics
      running_loss += loss.item()

    print(f'[{epoch + 1}] loss: {running_loss / 2000:.3f}')

  print('Finished Training')
  torch.save(model.state_dict(), path)

In [7]:
print("Device:", next(model.parameters()).device)
print("Batch size:", train_loader.batch_size)
print("Number of batches:", len(train_loader))

Device: cuda:0
Batch size: 64
Number of batches: 313


In [8]:
# train clean model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
train_model(model, attack = "clean", epochs = 30, prob = .1, k = 4, eps = 10./255, eps_step = (10./255)/4, trainloader = train_loader)

no input normalization
[1] loss: 0.274
[2] loss: 0.229
[3] loss: 0.195
[4] loss: 0.174
[5] loss: 0.159
[6] loss: 0.147
[7] loss: 0.136
[8] loss: 0.125
[9] loss: 0.117
[10] loss: 0.106
[11] loss: 0.097
[12] loss: 0.089
[13] loss: 0.082
[14] loss: 0.073
[15] loss: 0.067
[16] loss: 0.060
[17] loss: 0.054
[18] loss: 0.047
[19] loss: 0.041
[20] loss: 0.035
[21] loss: 0.029
[22] loss: 0.025
[23] loss: 0.021
[24] loss: 0.018
[25] loss: 0.018
[26] loss: 0.014
[27] loss: 0.013
[28] loss: 0.013
[29] loss: 0.011
[30] loss: 0.009
Finished Training


In [9]:
# Train pgd_linf with large ball
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
train_model(model, attack = "pgd_linf", epochs = 30, prob = .25, k = 4, eps = 10./255, eps_step = (10./255)/4, trainloader = train_loader)

no input normalization
[1] loss: 0.302
[2] loss: 0.276
[3] loss: 0.262
[4] loss: 0.255
[5] loss: 0.241
[6] loss: 0.240
[7] loss: 0.232
[8] loss: 0.225
[9] loss: 0.218
[10] loss: 0.212
[11] loss: 0.209
[12] loss: 0.193
[13] loss: 0.201
[14] loss: 0.189
[15] loss: 0.193
[16] loss: 0.180
[17] loss: 0.170
[18] loss: 0.173
[19] loss: 0.158
[20] loss: 0.163
[21] loss: 0.153
[22] loss: 0.152
[23] loss: 0.134
[24] loss: 0.145
[25] loss: 0.128
[26] loss: 0.128
[27] loss: 0.120
[28] loss: 0.119
[29] loss: 0.108
[30] loss: 0.111
Finished Training


# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [10]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1, steps = 1):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            x_batch = pgd_linf_untargeted(model, x_batch, y_batch, k = steps, eps = eps, eps_step = eps/steps)
        elif attack == 'pgd_l2':
            x_batch = pgd_l2_untargeted(model, x_batch, y_batch, k = steps, eps = eps, eps_step = eps/steps)
        else:
            pass

        # get the testing accuracy and update tot_test and tot_acc
        logits = model(x_batch)
        pred = torch.max(logits, dim=1)[1]
        tot_acc += (pred == y_batch).sum().item()
        tot_test += len(y_batch)

    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')
    return(tot_acc/tot_test, attack, eps)

In [11]:
# Test overall accuracy (no attack)
model.load_state_dict(torch.load('/content/clean_model.pt'))
test_model_on_single_attack(model, attack = "None")

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 57.71it/s]

Robust accuracy 0.74400 on None attack with eps = 0.1


(0.744, 'None', 0.1)

In [12]:
model.load_state_dict(torch.load('/content/pgd_linf_model.pt'))
test_model_on_single_attack(model, attack = "None")

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 58.86it/s]

Robust accuracy 0.71920 on None attack with eps = 0.1


(0.7192, 'None', 0.1)

## Single-Norm Robust Accuracy

In [13]:
# Various epsilon tests
for eps in range(12):
  model.load_state_dict(torch.load('/content/clean_model.pt'))
  # Evaluate on Linf attack with model 2 with eps = 8./255
  print(f"linf attack, eps = {eps}./255, clean")
  test_model_on_single_attack(model, attack='pgd_linf', eps = eps/255.0, steps = 4)
  model.load_state_dict(torch.load('/content/pgd_linf_model.pt'))
  # Evaluate on Linf attack with model 1 with eps = 8./255
  print(f"linf attack, eps = {eps}./255, pretr_linf")
  test_model_on_single_attack(model, attack='pgd_linf', eps = eps/255.0, steps = 4)
  print("\n")

linf attack, eps = 0./255, clean


Evaluating: 100%|██████████| 157/157 [00:08<00:00, 19.36it/s]


Robust accuracy 0.74400 on pgd_linf attack with eps = 0.0
linf attack, eps = 0./255, pretr_linf


Evaluating: 100%|██████████| 157/157 [00:07<00:00, 19.71it/s]


Robust accuracy 0.71920 on pgd_linf attack with eps = 0.0


linf attack, eps = 1./255, clean


Evaluating: 100%|██████████| 157/157 [00:07<00:00, 19.71it/s]


... (39 outputs left out)

Evaluating: 100%|██████████| 157/157 [00:07<00:00, 19.72it/s]


Robust accuracy 0.00000 on pgd_linf attack with eps = 0.043137254901960784
linf attack, eps = 11./255, pretr_linf


Evaluating: 100%|██████████| 157/157 [00:07<00:00, 19.74it/s]

Robust accuracy 0.17600 on pgd_linf attack with eps = 0.043137254901960784




In [14]:
# Various epsilon tests
for eps in range(12):
  model.load_state_dict(torch.load('/content/clean_model.pt'))
  # Evaluate on Linf attack with model 2 with eps = 8./255
  print(f"linf attack, eps = {eps}./255, clean")
  test_model_on_single_attack(model, attack='pgd_linf', eps = eps/255.0, steps = 1)
  model.load_state_dict(torch.load('/content/pgd_linf_model.pt'))
  # Evaluate on Linf attack with model 1 with eps = 8./255
  print(f"linf attack, eps = {eps}./255, pretr_linf")
  test_model_on_single_attack(model, attack='pgd_linf', eps = eps/255.0, steps = 1)
  print("\n")

linf attack, eps = 0./255, clean


Evaluating: 100%|██████████| 157/157 [00:04<00:00, 38.57it/s]


Robust accuracy 0.74400 on pgd_linf attack with eps = 0.0
linf attack, eps = 0./255, pretr_linf


Evaluating: 100%|██████████| 157/157 [00:04<00:00, 37.62it/s]


Robust accuracy 0.71920 on pgd_linf attack with eps = 0.0


linf attack, eps = 1./255, clean


Evaluating: 100%|██████████| 157/157 [00:04<00:00, 38.06it/s]


... (39 outputs left out)

Evaluating: 100%|██████████| 157/157 [00:04<00:00, 38.26it/s]


Robust accuracy 0.00980 on pgd_linf attack with eps = 0.043137254901960784
linf attack, eps = 11./255, pretr_linf


Evaluating: 100%|██████████| 157/157 [00:04<00:00, 37.84it/s]

Robust accuracy 0.20610 on pgd_linf attack with eps = 0.043137254901960784


